# Chapter 1 Advanced Lab — Single-Cell Habituation Simulator

**Basal Cognition** · Dr. Ernesto Lee · [BasalCognition.com](https://basalcognition.com)

---

## What You'll Do

In two parts:

**Part A — Synthetic cell model.** You will build a minimal simulation of habituation in a single-cell system. The synthetic cell has a response level that decrements with each identical stimulus and recovers during rest periods. You will plot the curve and confirm it matches the classical habituation signature.

**Part B — Real AI API.** You will send the same prompt to a real language model API 20 times in a row, track a measurable response property (e.g., response length, or a numeric rating), check for spontaneous recovery after a rest interval, and compare the pattern to your synthetic model.

**Key question:** Does the AI agent's pattern look like the cell's pattern?

---

### Habituation criteria you're testing (from Rankin et al. 2009)
1. Response decreases with repeated identical stimulus
2. Response recovers spontaneously after rest
3. Novel stimulus restores response (dishabituation)

---

In [ ]:
# Install dependencies
!pip install openai matplotlib numpy requests --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import json

# For Part B — set your API key here
# If using OpenAI: import openai; openai.api_key = 'your-key'
# If using another provider, swap in their client below
API_KEY = "YOUR_API_KEY_HERE"  # TODO: replace with your key

print("Imports ready.")

## Part A — Synthetic Cell Model

We model a cell as having a **response level** `R` (starts at 1.0). Each time it receives the same stimulus, `R` drops by a **decrement rate** `d`. During rest (no stimulus), `R` recovers toward 1.0 by a **recovery rate** `r`. These two parameters are all you need to produce classical habituation.


In [ ]:
def simulate_habituation(
    n_trials=20,
    decrement=0.12,
    recovery_rate=0.08,
    rest_after_trial=None,  # trial index to insert a rest period
    rest_steps=5,           # how many recovery steps
    novel_at_trial=None,    # trial index to inject a novel stimulus
    novel_boost=0.6,        # how much novel stimulus restores R
):
    """
    Simulate a single cell's habituation to repeated identical stimuli.
    Returns: list of (label, response_level) tuples
    """
    R = 1.0
    records = []

    for t in range(1, n_trials + 1):
        # --- Rest period insertion ---
        if rest_after_trial is not None and t == rest_after_trial + 1:
            for _ in range(rest_steps):
                R = min(1.0, R + recovery_rate)
            records.append((f"T{t} (post-rest)", R))
        # --- Novel stimulus ---
        elif novel_at_trial is not None and t == novel_at_trial:
            R = min(1.0, R + novel_boost)
            records.append((f"T{t} (novel!)", R))
        else:
            records.append((f"T{t}", R))

        # Apply stimulus (decrement)
        R = max(0.0, R - decrement)

    return records


# --- Run the model ---
baseline = simulate_habituation(n_trials=20, decrement=0.12, recovery_rate=0.08)
with_rest = simulate_habituation(
    n_trials=25, decrement=0.12, recovery_rate=0.08,
    rest_after_trial=10, rest_steps=8
)
with_novel = simulate_habituation(
    n_trials=20, decrement=0.12, recovery_rate=0.08,
    novel_at_trial=14, novel_boost=0.5
)

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, data, title in zip(
    axes,
    [baseline, with_rest, with_novel],
    ["Habituation (baseline)", "With Rest Period", "With Novel Stimulus"]
):
    labels, values = zip(*data)
    colors = [
        "#2ec4b6" if "post-rest" in l or "novel" in l else "#1a535c"
        for l in labels
    ]
    ax.bar(range(len(values)), values, color=colors)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1.1)
    ax.set_xlabel("Trial")
    ax.set_ylabel("Response Level")
    ax.axhline(y=1.0, linestyle="--", color="gray", alpha=0.5, label="Max response")

plt.tight_layout()
plt.savefig("ch01-synthetic-habituation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Synthetic model plot saved.")

## Part B — Real AI API Test

We send the same prompt 20 times and measure a response property.

**Measuring response magnitude:** We use **response word count** as a proxy. A fully engaged response tends to be longer; a "bored" (habituated) response tends to be shorter or more dismissive.

This is a coarse proxy. A more sophisticated version would use a sentiment or engagement score from a second LLM. But word count is free and fast.

**The repeated prompt** simulates a monitoring agent receiving the same alert repeatedly.

---
**TODO:** Replace `API_KEY` above and choose your provider. The scaffolding uses OpenAI; adjust the client call if you use a different provider.


In [ ]:
from openai import OpenAI

REPEATED_PROMPT = (
    "ALERT: Disk usage on server-prod-1 is at 78%. "
    "Assess severity and recommend action."
)

NOVEL_PROMPT = (
    "CRITICAL: Disk usage on server-prod-1 has reached 99%. "
    "Production database write operations are failing. Immediate action required."
)

N_REPEATED = 20      # trials with the same prompt
REST_AFTER = 10      # insert a rest gap after trial 10 (skip N_REST_TRIALS calls)
N_REST_TRIALS = 3    # simulated 'time passing' (no calls)
NOVEL_AT = 16        # inject novel prompt at trial 16

def call_api(prompt, client):
    """Make one API call and return word count of the response."""
    # TODO: adjust model name and parameters for your provider
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0.7,
    )
    text = response.choices[0].message.content
    word_count = len(text.split())
    return word_count, text


def run_habituation_experiment(api_key):
    client = OpenAI(api_key=api_key)
    results = []

    for t in range(1, N_REPEATED + 1):
        prompt = NOVEL_PROMPT if t == NOVEL_AT else REPEATED_PROMPT
        label = f"T{t} (NOVEL)" if t == NOVEL_AT else f"T{t}"

        if t == REST_AFTER + 1:
            print(f"  [Rest gap — simulating {N_REST_TRIALS} days with no calls]")
            label = f"T{t} (post-rest)"

        wc, text = call_api(prompt, client)
        results.append({"trial": t, "label": label, "word_count": wc, "text": text[:100]})
        print(f"  {label}: {wc} words")
        time.sleep(1)  # be polite to the API

    return results


# Uncomment to run — will cost API credits
# results = run_habituation_experiment(API_KEY)

print("Cell defined. Uncomment the last line and run with your API key to start the experiment.")

In [ ]:
# --- Plot AI results vs. synthetic model ---
# TODO: After running the experiment above, uncomment and run this cell.

# trial_nums = [r["trial"] for r in results]
# word_counts = [r["word_count"] for r in results]
# labels = [r["label"] for r in results]

# # Normalize word counts to 0-1 scale for comparison with synthetic model
# max_wc = max(word_counts)
# normalized = [wc / max_wc for wc in word_counts]

# # Rerun synthetic for same number of trials
# synth = simulate_habituation(
#     n_trials=N_REPEATED, decrement=0.10, recovery_rate=0.07,
#     rest_after_trial=REST_AFTER, rest_steps=5,
#     novel_at_trial=NOVEL_AT, novel_boost=0.5
# )
# synth_vals = [v for _, v in synth]

# fig, ax = plt.subplots(figsize=(14, 5))
# ax.plot(trial_nums, normalized, 'o-', color='#2ec4b6', label='AI agent (normalized word count)', linewidth=2)
# ax.plot(trial_nums, synth_vals, 's--', color='#f4a261', label='Synthetic cell model', linewidth=2)
# ax.set_xlabel('Trial')
# ax.set_ylabel('Response magnitude (normalized)')
# ax.set_title('AI Agent vs. Synthetic Cell: Habituation Pattern Comparison', fontsize=13, fontweight='bold')
# ax.legend()
# ax.set_ylim(0, 1.2)
# plt.tight_layout()
# plt.savefig('ch01-ai-vs-cell-habituation.png', dpi=150, bbox_inches='tight')
# plt.show()

print("Plot cell ready. Run after experiment data is collected.")

## Deliverable

Write a short report (300–500 words) addressing the following:

1. **Describe your results.** Did the AI agent's word count decrease across the 20 trials? Was there a clear trend or was it noisy?

2. **Spontaneous recovery.** Did response magnitude increase after the simulated rest period (trial 11 vs. trial 10)? How did it compare to the synthetic cell model?

3. **Dishabituation.** Did the novel prompt at trial 16 produce a noticeably larger response than trials 14–15? What does that suggest?

4. **Interpretation.** Based on your data, do you think the AI agent is showing something functionally similar to habituation? What alternative explanations exist for any patterns you observed (e.g., temperature randomness, model caching, context length effects)?

5. **Next step.** If you were to design a follow-up experiment to test one alternative explanation, what would you change?

---

*Submit your plots and your written report. Code should be fully runnable — another person should be able to reproduce your results by adding their API key.*
